# Delta Lake Optimization

- Delta Lake's OPTIMIZE command is used to compact small files into larger ones <br>
- After optimizing, we use the VACUUM command to clean up obsolete files

Data source: https://excelbianalytics.com/downloads-17-sample-csv-files-data-sets-for-testing-credit-card/

In [0]:
%sql
CREATE CATALOG IF NOT EXISTS main;
USE CATALOG main;
CREATE SCHEMA IF NOT EXISTS default;
CREATE VOLUME IF NOT EXISTS default.raw_data;

In [0]:
# Loading the 2M rows file
file_path = '/Volumes/main/default/raw_data/2m CC Records.csv'
df_cc = spark.read.format('csv').option('header', 'true').load(file_path)

# Print colummn name and row count
total_rows = df_cc.count()
print('Column names:', df_cc.columns)
print('Total records in the dataset: ', total_rows)

# Mapping column names
from pyspark.sql.functions import col
columns_mapping = {
    'Card Type Code': 'card_type_code', 
    'Card Type Full Name':'card_type_fullname', 
    'Issuing Bank':'issuing_bank', 
    'Card Number':'card_number', 
    "Card Holder's Name":'card_holder_name', 
    'CVV/CVV2':'cvv_cvv2', 
    'Issue Date':'issue_date', 
    'Expiry Date':'expiry_date', 
    'Billing Date':'billing_date', 
    'Card PIN':'card_ping', 
    'Credit Limit':'credit_limit'
}

# Renaming columns
for old_name, new_name in columns_mapping.items():
    df_cc = df_cc.withColumnRenamed(old_name, new_name)

# Creating delta table
df_cc.write.mode('overwrite').saveAsTable('main.default.creditcards')


In [0]:
df_loaded = spark.read.table('main.default.creditcards')
display(df_loaded)

In [0]:
%sql
-- Optimize
OPTIMIZE main.default.creditcards
ZORDER by (`issuing_bank`);

-- Vacuum table
VACUUM main.default.creditcards RETAIN 168 HOURS;